<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-11-self-hosting/lesson-11.1-gemma-cloud-run/notebooks/GCP_Capstone_11.1_GemmaCloudRun.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11.1 Deploy Gemma on Cloud Run L4 GPU — vLLM + Scale-to-Zero
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
# Configuration for Gemma on Cloud Run
PROJECT_ID = 'your-project-id'
REGION = 'us-central1'
REPO = 'vllm-repo'
IMAGE_NAME = 'gemma-3-4b-it'
SERVICE = 'gemma-vllm'
MODEL = 'google/gemma-3-4b-it'

print(f'Deployment plan:')
print(f'  Project:  {PROJECT_ID}')
print(f'  Region:   {REGION} (L4 GA region)')
print(f'  Image:    {REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{IMAGE_NAME}:latest')
print(f'  Service:  {SERVICE}')
print(f'  Model:    {MODEL}')


## Cell 1: Production Dockerfile


In [ ]:
DOCKERFILE = '''FROM vllm/vllm-openai:v0.16.0

ENV HF_HOME=/model-cache
ARG HF_TOKEN

# Bake weights into image for fast cold start
RUN huggingface-cli login --token ${HF_TOKEN} && \\
    huggingface-cli download google/gemma-3-4b-it

# Block runtime network calls
ENV HF_HUB_OFFLINE=1

EXPOSE 8080

ENTRYPOINT python3 -m vllm.entrypoints.openai.api_server \\
    --port ${PORT:-8080} \\
    --model google/gemma-3-4b-it \\
    --dtype bfloat16 \\
    --gpu-memory-utilization 0.90 \\
    --max-model-len 4096 \\
    --max-num-seqs 64 \\
    --cuda-graph-sizes 1,2,4,8,16,32,64 \\
    --enable-prefix-caching
'''

with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)
print('Dockerfile written')
print(f'Base: vllm/vllm-openai (~4.8GB)')
print(f'+ Gemma 3 4B weights (~8GB)')
print(f'= Final image ~13-15GB')


## Cell 2: Cloud Build Config (HF_TOKEN from Secret Manager)


In [ ]:
CLOUDBUILD = f'''steps:
- name: 'gcr.io/cloud-builders/docker'
  entrypoint: 'bash'
  secretEnv: ['HF_TOKEN_SECRET']
  args:
  - '-c'
  - |
      docker build \\
        --build-arg HF_TOKEN=$$HF_TOKEN_SECRET \\
        -t {REGION}-docker.pkg.dev/$PROJECT_ID/{REPO}/{IMAGE_NAME}:latest .
- name: 'gcr.io/cloud-builders/docker'
  args: ['push', '{REGION}-docker.pkg.dev/$PROJECT_ID/{REPO}/{IMAGE_NAME}:latest']
images: ['{REGION}-docker.pkg.dev/$PROJECT_ID/{REPO}/{IMAGE_NAME}:latest']
options:
  machineType: E2_HIGHCPU_32
availableSecrets:
  secretManager:
  - versionName: projects/$PROJECT_ID/secrets/HF_TOKEN/versions/latest
    env: 'HF_TOKEN_SECRET'
'''

with open('cloudbuild.yaml', 'w') as f:
    f.write(CLOUDBUILD)
print('cloudbuild.yaml written')
print()
print('Build command:')
print(f'  gcloud builds submit --config=cloudbuild.yaml --project={PROJECT_ID}')
print()
print('Expected build time: ~15-30 min on E2_HIGHCPU_32')


## Cell 3: Deployment Command with Three Critical Flags


In [ ]:
deploy_cmd = f'''gcloud run deploy {SERVICE} \\
  --image {REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{IMAGE_NAME}:latest \\
  --region {REGION} \\
  --service-account vllm-sa@{PROJECT_ID}.iam.gserviceaccount.com \\
  --gpu 1 \\
  --gpu-type nvidia-l4 \\
  --cpu 8 \\
  --memory 32Gi \\
  --port 8000 \\
  --max-instances 3 \\
  --min-instances 0 \\
  --concurrency 64 \\
  --timeout 600 \\
  --no-cpu-throttling \\
  --no-gpu-zonal-redundancy \\
  --cpu-boost \\
  --no-allow-unauthenticated \\
  --startup-probe=httpGet.path=/health,httpGet.port=8000,initialDelaySeconds=120,failureThreshold=5,timeoutSeconds=10,periodSeconds=30
'''
print(deploy_cmd)
print()
print('THREE CRITICAL FLAGS:')
print('  --no-cpu-throttling        (vLLM needs CPU continuously, not just startup)')
print('  --no-gpu-zonal-redundancy  (cuts $1.047 to $0.672/hr = 40% savings)')
print('  --cpu-boost                (2x CPU during startup, cuts minutes off cold start)')


## Cell 4: Python Client with ID Token Auth


In [ ]:
# Template Python client for authenticated Cloud Run endpoint
PYTHON_CLIENT = '''
import google.oauth2.id_token
import google.auth.transport.requests
from openai import OpenAI
import time

SERVICE_URL = "https://gemma-vllm-xxxxx-uc.a.run.app"

def get_authenticated_client(service_url: str) -> OpenAI:
    """Returns OpenAI client configured for private Cloud Run."""
    request = google.auth.transport.requests.Request()
    id_token = google.oauth2.id_token.fetch_id_token(request, service_url)
    
    return OpenAI(
        base_url=f"{service_url}/v1",
        api_key="not-needed",
        default_headers={"Authorization": f"Bearer {id_token}"},
    )

# DocuMind classification example
def classify_document(text: str) -> str:
    client = get_authenticated_client(SERVICE_URL)
    response = client.chat.completions.create(
        model="google/gemma-3-4b-it",
        messages=[
            {"role": "system", "content": "Classify as INVOICE, CONTRACT, REPORT, or RECEIPT. Reply ONE word."},
            {"role": "user", "content": text},
        ],
        max_tokens=10,
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()

# DocuMind streaming Q&A
def stream_answer(question: str):
    client = get_authenticated_client(SERVICE_URL)
    stream = client.chat.completions.create(
        model="google/gemma-3-4b-it",
        messages=[{"role": "user", "content": question}],
        stream=True,
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
'''
print(PYTHON_CLIENT)


## Cell 5: Health Check Poller (Cold Start Resilience)


In [ ]:
# Production pattern: poll /health before first request during cold start
import time
import requests

def wait_for_ready(service_url: str, id_token: str,
                    max_attempts: int = 20, base_delay: float = 2.0) -> bool:
    '''Exponential backoff health poll.
    
    Gemma 3 4B cold start ~19s. Gemma 9B ~35s. This poller handles both.
    Returns True when /health returns 200, False after max_attempts.
    '''
    headers = {'Authorization': f'Bearer {id_token}'}
    for attempt in range(max_attempts):
        try:
            r = requests.get(f'{service_url}/health', headers=headers, timeout=5)
            if r.status_code == 200:
                print(f'  Ready after {attempt + 1} attempts')
                return True
        except requests.RequestException:
            pass
        delay = min(base_delay * (1.5 ** attempt), 30.0)
        print(f'  Attempt {attempt + 1}: not ready, waiting {delay:.1f}s')
        time.sleep(delay)
    return False

print('wait_for_ready() function ready')
print('Call BEFORE first request when min-instances=0 to avoid timeouts during cold start')


## Cell 6: Cost Calculator — Self-Hosted vs Vertex AI


In [ ]:
# April 2026 pricing
L4_GPU_HR        = 0.672    # no zonal redundancy
CPU_8VCPU_HR     = 0.518
MEM_32GIB_HR     = 0.230
INSTANCE_HR      = L4_GPU_HR + CPU_8VCPU_HR + MEM_32GIB_HR  # ~$1.42/hr

# Gemini API pricing (per 1M tokens, April 2026)
FLASH_IN, FLASH_OUT = 1.50, 7.50
PRO_IN, PRO_OUT     = 2.00, 12.00

def cloud_run_cost(tokens_per_day: int, hours_per_day: int = 8,
                    instances_needed: int = 1):
    '''Cost of self-hosting Gemma on Cloud Run L4.'''
    days_per_month = 30
    hrs_per_month = hours_per_day * days_per_month * instances_needed
    return INSTANCE_HR * hrs_per_month

def gemini_cost(tokens_per_day: int, model: str = 'flash'):
    '''Cost of Gemini API assuming 80% input, 20% output split.'''
    days_per_month = 30
    total_tokens_per_month = tokens_per_day * days_per_month
    input_tokens = total_tokens_per_month * 0.8
    output_tokens = total_tokens_per_month * 0.2
    if model == 'flash':
        return (input_tokens * FLASH_IN + output_tokens * FLASH_OUT) / 1_000_000
    elif model == 'pro':
        return (input_tokens * PRO_IN + output_tokens * PRO_OUT) / 1_000_000

# Compare across volumes
print(f'{"Volume":<15} {"Cloud Run L4":>15} {"Gemini Flash":>15} {"Gemini Pro":>15}')
print('-' * 65)
for volume in [2_000_000, 10_000_000, 50_000_000, 100_000_000]:
    # Scale instances for >25M tokens/day
    instances = max(1, volume // 25_000_000)
    cr = cloud_run_cost(volume, hours_per_day=8, instances_needed=instances)
    gf = gemini_cost(volume, 'flash')
    gp = gemini_cost(volume, 'pro')
    vol_str = f'{volume // 1_000_000}M/day'
    print(f'{vol_str:<15} ${cr:>12,.0f} ${gf:>12,.0f} ${gp:>12,.0f}')

print()
print('Break-even: Cloud Run beats Gemini Flash around ~4M tokens/day')
print('Cloud Run beats Gemini Pro above ~3M tokens/day. Up to ~9x cheaper at 50M.')

## Cell 7: Load Testing Pattern


In [ ]:
# DocuMind load test: measure throughput and latency under concurrent load
LOAD_TEST = '''
import asyncio
import time
from openai import AsyncOpenAI

async def run_load_test(service_url: str, id_token: str,
                         n_requests: int = 100, concurrency: int = 16):
    """Fire n_requests with concurrency workers. Measure latency + throughput."""
    client = AsyncOpenAI(
        base_url=f"{service_url}/v1",
        api_key="not-needed",
        default_headers={"Authorization": f"Bearer {id_token}"},
    )
    
    async def one_request(idx: int):
        start = time.time()
        response = await client.chat.completions.create(
            model="google/gemma-3-4b-it",
            messages=[{"role": "user", "content": f"Classify doc #{idx}: ..."}],
            max_tokens=50,
        )
        duration = time.time() - start
        tokens_out = response.usage.completion_tokens
        return duration, tokens_out
    
    sem = asyncio.Semaphore(concurrency)
    async def bounded(idx):
        async with sem:
            return await one_request(idx)
    
    start = time.time()
    results = await asyncio.gather(*[bounded(i) for i in range(n_requests)])
    elapsed = time.time() - start
    
    latencies = [r[0] for r in results]
    total_tokens = sum(r[1] for r in results)
    
    print(f"Total: {n_requests} requests in {elapsed:.1f}s")
    print(f"Throughput: {n_requests/elapsed:.1f} req/s, {total_tokens/elapsed:.1f} tok/s")
    print(f"Latency p50: {sorted(latencies)[len(latencies)//2]:.2f}s")
    print(f"Latency p95: {sorted(latencies)[int(len(latencies)*0.95)]:.2f}s")
    print(f"Latency p99: {sorted(latencies)[int(len(latencies)*0.99)]:.2f}s")
'''
print(LOAD_TEST)


## Cell 8: Three DocuMind Production Patterns


In [ ]:
# Pattern 1: Scale-to-zero batch processing
PATTERN_1 = {
    'name': 'Scale-to-zero batch',
    'use_case': 'Nightly document classification on 50K accumulated uploads',
    'config': {
        'min-instances': 0,
        'max-instances': 5,
        'concurrency': 64,
    },
    'cost_profile': '$0 idle, ~$250/month for 8hr business-hour processing',
    'tradeoff': 'Cold start 19-35s acceptable for batch workloads',
}

# Pattern 2: Warm instance interactive
PATTERN_2 = {
    'name': 'Warm instance interactive',
    'use_case': 'User-facing document Q&A, sub-200ms first token required',
    'config': {
        'min-instances': 1,
        'max-instances': 3,
        'concurrency': 32,
    },
    'cost_profile': '~$756/month 24/7 for warm instance',
    'tradeoff': 'Higher cost but no cold start. Fallback to Gemini Flash on overflow',
}

# Pattern 3: Multi-model constellation
PATTERN_3 = {
    'name': 'Multi-model constellation on ONE L4',
    'use_case': 'DocuMind pipeline: classify -> extract -> summarize',
    'models': {
        'classifier':  'Gemma 2B INT4  (~2GB)',
        'extractor':   'Gemma 3 4B BF16 (~8GB)',
        'summarizer':  'Gemma 2 9B INT8 (~9GB)',
    },
    'total_vram': '~19GB on 24GB L4 (plenty of KV cache)',
    'cost_profile': '~$1.42/hr ONE instance vs 3 separate Vertex AI endpoints',
    'tradeoff': 'Single point of failure, more complex routing, massive cost savings',
}

for p in [PATTERN_1, PATTERN_2, PATTERN_3]:
    print(f"=== {p['name']} ===")
    for k, v in p.items():
        if k != 'name':
            print(f'  {k}: {v}')
    print()


## Done!
- Production Dockerfile with baked Gemma weights
- Cloud Build + Secret Manager for HF_TOKEN
- Deployment with three critical flags
- Python OpenAI SDK client with ID token auth
- Health check poller for cold start resilience
- Cost calculator across 2M / 10M / 50M / 100M tokens/day
- Async load testing pattern
- Three production architecture patterns
